# Simplest maze solutions

The Colab version of this code can be used with the below commands. The code can also be downloaded locally and run as a Python Jupyter notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Useful imports:

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from enum import Enum
import random
import gymnasium as gym
from gymnasium import spaces
from gymnasium.envs.registration import register
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")

Run the baseline of the environment in the cell below.

In [6]:
class Actions(Enum):
  RIGHT = 0
  UP = 1
  LEFT = 2
  DOWN = 3

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array", "rgb_array_list"], "render_fps": 4}

    def __init__(self, hole_map, render_mode=None, size=3, max_steps=100):
        self.size = size  # The size of the square grid
        self.max_steps = max_steps
        self.current_step = 0

        self._agent_location = np.array([-1, -1], dtype=np.int64)
        self._target_location = np.array([-1, -1], dtype=np.int64)

        # Add the hole map
        if hole_map is None:
           self.holes = np.zeros((size, size), dtype=np.int64)
        else:
           self.holes = np.array(hole_map, dtype=np.int64)
           self.size = self.holes.shape[0]  # Sync size with hole_map dimensions

        # Observations are dictionaries with the agent's and the target's location.
        # Each location is encoded as an element of {0, ..., 'size'}^2, i.e. MultiDiscrete([size, size]).
        self.observation_space = spaces.Dict(
            {
                "agent": spaces.Box(0, size-1, shape=(2,), dtype=np.int64),
                "target": spaces.Box(0, size-1, shape=(2,), dtype=np.int64),
                "holes": spaces.Box(0, 1, shape=(size, size), dtype=np.int64)
            }
        )

        # We have 4 actions, corresponding to "right", "up", "left", "down"
        self.action_space = spaces.Discrete(4)

        """
        The following dictionary maps abstract actions from 'self.action_space' to
        the direction we will walk in if that action is taken.
        i.e. 0 corresponds to "right", 1 to "up" etc.
        Uses NumPy [row, col] convention where row 0 is at the top.
        """
        self._action_to_direction = {
            Actions.RIGHT.value: np.array([0, 1], dtype=np.int64),
            Actions.UP.value: np.array([-1, 0], dtype=np.int64),
            Actions.LEFT.value: np.array([0, -1], dtype=np.int64),
            Actions.DOWN.value: np.array([1, 0], dtype=np.int64),
        }

        assert render_mode is None or render_mode in self.metadata["render_modes"]
        self.render_mode = render_mode

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0

        # Ensure valid target location relative to current grid size
        self._target_location = np.array([self.size - 1, self.size - 1], dtype=np.int64)

        while True:
            pos = self.np_random.integers(0, self.size, size=2, dtype=np.int64)
            if self.holes[pos[0], pos[1]] == 0:
                self._agent_location = pos
                break

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "rgb_array":
            self._render_frame()

        return observation, info

    def step(self, action):
        self.current_step += 1
        direction = self._action_to_direction[action]
        # 'np.clip' makes sure that the agent does not leave the grid
        new_position = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # prevent movement into holes
        if self.holes[new_position[0], new_position[1]] == 0:
          self._agent_location = new_position

        # An episode is done iff the agent has reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)
        truncated = self.current_step >= self.max_steps
        # Assign rewards
        if terminated:
          reward = 1.0
        elif self.holes[new_position[0], new_position[1]] == 1:
          reward = -1.0
        else :
          reward = 0.0

        observation = self._get_obs()
        info = self._get_info()

        if self.render_mode == "rgb_array":
            self._render_frame()

        return observation, reward, terminated, truncated, info

    def _get_obs(self):
        return {"agent": self._agent_location,
                "target": self._target_location,
                "holes": self.holes}

    def _get_info(self):
        return {
            "distance": np.linalg.norm(self._agent_location - self._target_location, ord=1)
        }

    def render(self):
        if self.render_mode in ["rgb_array", "rgb_array_list"]:
            return self._render_frame()
        return None

    def _render_frame(self):
        size = self.size
        # Create a blank RGB canvas (light blue)
        canvas = np.ones((size, size, 3), dtype=np.uint8) * [0, 255, 255]

        # Draw holes (blue)
        canvas[self.holes == 1] = [0, 0, 225]

        # Draw target (green)
        canvas[self._target_location[0], self._target_location[1]] = [0, 128, 0]

        return canvas.astype(np.uint8)

    def close(self):
        pass

In [7]:
# Register the environment of size 3
gym.register(
     id="GridWorld-v1",
     entry_point="__main__:GridWorldEnv",
)

hole_map = np.array([
     [0,1,0],
     [0,1,1],
     [0,0,0],
])

env = gym.make(
   "GridWorld-v1",
   hole_map=hole_map,
   render_mode="rgb_array_list",
   size=3
)

Pre-run the functions to display the agent steps path and the annimation in the environment.

In [8]:
def render_policy_path(q_table, start_pos=(0,0), env_name="GridWorld-v1", hole_map=hole_map, max_steps=100):
    """
    Captures all frame images of the optimal policy for the annimation.
    """
    # Create a visual-ready environment instance
    eval_env = gym.make(
        env_name,
        hole_map=hole_map,
        render_mode="rgb_array_list",
        size=3
    )

    # Extract the raw underlying environment class instance
    raw_env = eval_env.unwrapped

    # Save the original rendering function so we can restore it later
    original_render_frame = raw_env._render_frame

    # Define our patched rendering function that paints the agent gold
    def custom_render_frame():
        # Call the original maze renderer to get the base canvas
        canvas = original_render_frame()

        # Paint the agent gold [255, 215, 0] at its current environment position
        agent_r, agent_c = raw_env._agent_location
        canvas[agent_r, agent_c] = [255, 215, 0]

        # Upscale grid (e.g. 5x5 -> 250x250 pixels) so Matplotlib displays it clearly
        scale = 50
        upscaled_canvas = np.repeat(np.repeat(canvas, scale, axis=0), scale, axis=1)
        return upscaled_canvas

    # Monkey-patch the environment method with our custom renderer
    raw_env._render_frame = custom_render_frame

    obs, _ = eval_env.reset(seed=42) # change fixed seed for testing
    if start_pos is not None:
      raw_env._agent_location = np.array(start_pos, dtype=np.int32)
      state = start_pos
    else:
      state = get_coord(obs)

    frames = [eval_env.render()]

    done = False
    step = 0

    while not done and step < max_steps:
        # Pick the optimal action learned in Q-table
        action = np.argmax(q_table[state[0], state[1]])

        next_obs, reward, terminated, truncated, info = eval_env.step(action)
        state = get_coord(next_obs)
        done = terminated or truncated

        # Save the frame
        frame = eval_env.render()
        if frame is not None:
            frames.append(frame)

        step += 1

    # Restore the original method before closing
    raw_env._render_frame = original_render_frame
    eval_env.close()
    return frames

def plot_maze_with_arrows(q_table, title="Learned Path"):
    obs, _ = env.reset(seed=42) # change fixed seed for testing

    # Force agent start to (0,0) for evaluation
    env.unwrapped._agent_location = np.array([0, 0], dtype=int)
    state = (0, 0)
    target_state = (env.unwrapped.size - 1, env.unwrapped.size - 1)

    # Mapping actions to display arrows
    arrow_map = {0: "→", 1: "↑", 2: "←", 3: "↓"}

    # Render base maze frame
    frame = env.unwrapped._render_frame()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(frame)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis('off')

    steps = 0
    path_arrows = []
    max_steps = 100
    done = False

    while state != target_state and not done and steps < max_steps:
        # Pick the optimal action learned in Q-table
        action = int(np.argmax(q_table[state[0], state[1]]))
        arrow = arrow_map[action]
        path_arrows.append(arrow)

        ax.text(
            state[1], state[0], arrow,
            color="black", fontsize=20, fontweight="bold",
            ha="center", va="center"
        )

        next_obs, reward, terminated, truncated, info = env.step(action)

        state = (int(next_obs["agent"][0]), int(next_obs["agent"][1]))
        done = terminated or truncated
        steps += 1

    plt.show()
    return steps, " -> ".join(path_arrows)

def print_q_table(q_table, name, hole_map):
    action_names = ["RIGHT (→)", "UP (↑)", "LEFT (←)", "DOWN (↓)"]
    print(f"\n=============================================")
    print(f"          Q-TABLE FOR {name.upper()}         ")
    print(f"=============================================")

    # Print the column headers
    print(f"{'State (row, column)':<14} | {'State Type':<12} | " + " | ".join([f"{act:<10}" for act in action_names]))
    print("-" * 78)

    rows, cols, _ = q_table.shape
    for r in range(rows):
        for c in range(cols):
            # Check if this state is a hole or wall based on your map
            is_hole = hole_map[r, c] == 1
            state_type = "Hole (Wall)" if is_hole else "Clear Path"

            # Extract values for the 4 actions at this specific coordinate
            q_values = q_table[r, c]

            # Format numbers cleanly to 4 decimal places
            v_str = " | ".join([f"{v:10.4f}" for v in q_values])

            print(f"({r}, {c}){' ':<8} | {state_type:<12} | {v_str}")

def plot_maze_with_qvalues(q_table, title="Optimal q-values"):
    obs, _ = env.reset(seed=42) # change fixed seed for testing

    target_state = (env.unwrapped.size - 1, env.unwrapped.size - 1)

    # Render base maze frame
    frame = env.unwrapped._render_frame()

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(frame)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis('off')

    steps = 0
    max_steps = 100
    done = False

    for r in range(grid_size):
        for c in range(grid_size):
            best_qvalues = np.max(q_table[r, c])
            qvalues = f"{best_qvalues:.2f}"

            ax.text(
                c, r, qvalues,
                color="black", fontsize=12, fontweight="bold",
                ha="center", va="center"
            )

    plt.show()

def display_animation(frames, title="Agent Path Animation", interval=300):
    """
    Renders an inline playable video inside your Jupyter notebook.
    """
    fig, ax = plt.subplots(figsize=(6, 6))
    plt.axis("off")
    plt.title(title, fontsize=14, fontweight="bold")

    im = ax.imshow(frames[0])

    def update(frame_idx):
        im.set_array(frames[frame_idx])
        return [im]

    anim = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval, # time between frames in milliseconds
        blit=True,
        repeat=False
    )

    plt.close(fig) # Prevent duplicate static figure display
    display(HTML(anim.to_html5_video()))

Q-learning and SARSA are implemented below.

Training hyperparameters and epsilon-greedy policy:

In [9]:
# Introduce epsilon decay to lower the loop risk
def compute_epsilon_decay(epsilon_start, epsilon_min, target_episodes):
    return (epsilon_min / epsilon_start) ** (1.0 / target_episodes)

episodes = 100
epsilon_start = 1.0
epsilon_min = 0.01
epsilon_decay = compute_epsilon_decay(epsilon_start, epsilon_min, target_episodes=episodes)

alpha = 0.1  # Learning rate
gamma = 0.9  # Discount factor

# Initialize Q-Tables with matching dimensions: (Rows, Cols, Actions)
grid_size = env.unwrapped.size
action_size = env.action_space.n
q_table_ql = np.zeros((grid_size, grid_size, action_size))
q_table_sarsa = np.zeros((grid_size, grid_size, action_size))

# Helper function to reliably get coordinates from Gym dict observation
def get_coord(obs):
    # obs can be a tuple (obs, info) from reset() or direct dict from step()
    actual_obs = obs[0] if isinstance(obs, tuple) else obs
    return int(actual_obs["agent"][0]), int(actual_obs["agent"][1])

def choose_action(state_coord, q_table, epsilon, env):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample() # Explore
    else:
        return np.argmax(q_table[state_coord[0], state_coord[1]]) # Exploit

Q-learning implementation: (while loop has to be edited)

In [ ]:
print("Running Q-Learning...")

epsilon = epsilon_start
for episode in range(episodes):
    obs, info = env.reset()
    state = get_coord(obs)
    done = False

    while not done:
        action = choose_action(state, q_table_ql, epsilon, env)
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = get_coord(next_obs)
        done = terminated or truncated

        # Off-policy update using max Q(s', a')
        best_next_action = np.argmax(q_table_ql[next_state[0], next_state[1]])
        td_target = reward + gamma * q_table_ql[next_state[0], next_state[1], best_next_action] * (not terminated)
        td_error = td_target - q_table_ql[state[0], state[1], action]

        q_table_ql[state[0], state[1], action] += alpha * td_error
        state = next_state
    epsilon = max(epsilon_min, epsilon * epsilon_decay)   # Exploration rate

Different results of the Q-leanring algorithm:

In [ ]:
# --- Q-Learning optimal path ---
ql_steps, ql_path = plot_maze_with_arrows(q_table_ql, "Q-Learning Optimal Path")

In [ ]:
print_q_table(q_table_ql, "Q-Learning", hole_map)

In [ ]:
# --- Q-Learning optimal path ---
plot_maze_with_qvalues(q_table_ql, "Q-Learning Optimal Path")

In [ ]:
ql_frames = render_policy_path(q_table_ql)
display_animation(ql_frames, title="Q-Learning Optimal Path")

SARSA implementation:

In [ ]:
print("Running SARSA...")
for episode in range(episodes):
    obs = env.reset()
    state = get_coord(obs)
    action = choose_action(state, q_table_sarsa, epsilon, env)
    done = False

    while not done:
        next_obs, reward, terminated, truncated, info = env.step(action)
        next_state = get_coord(next_obs)
        done = terminated or truncated

        # On-policy action choice
        next_action = choose_action(next_state, q_table_sarsa, epsilon, env)

        # Update using the actual chosen next action
        td_target = reward + gamma * q_table_sarsa[next_state[0], next_state[1], next_action] * (not terminated)
        td_error = td_target - q_table_sarsa[state[0], state[1], action]

        q_table_sarsa[state[0], state[1], action] += alpha * td_error

        state = next_state
        action = next_action

    epsilon = max(epsilon_min, epsilon * epsilon_decay)   # Exploration rate

Different results of the SARSA algorithm:

In [ ]:
# --- SARSA optimal path ---
sarsa_steps, sarsa_path = plot_maze_with_arrows(q_table_sarsa, "SARSA Optimal Path")

In [ ]:
print_q_table(q_table_sarsa, "SARSA", hole_map)

In [ ]:
# --- SARSA optimal path ---
plot_maze_with_qvalues(q_table_sarsa, "SARSA Optimal Path")

In [ ]:
sarsa_frames = render_policy_path(q_table_sarsa)
display_animation(sarsa_frames, title="SARSA Optimal Path")

In [20]:
def train_agent(algorithm):
    q_table = np.zeros((grid_size, grid_size, action_size))
    total_steps = 0

    step_history = []
    reward_history = []

    for episode in range(episodes):
        obs = env.reset()
        state = get_coord(obs)
        done = False
        episode_reward = 0

        action = choose_action(state, q_table, epsilon, env)

        while not done:
            next_obs, reward, terminated, truncated, _ = env.step(action)
            next_state = get_coord(next_obs)
            done = terminated or truncated

            total_steps += 1
            episode_reward += reward

            if algorithm == "q_learning":
                # Off-policy update using max Q(s', a')
                best_next_action = np.argmax(q_table[next_state[0], next_state[1]])
                td_target = reward + gamma * q_table[next_state[0], next_state[1], best_next_action] * (not terminated)
                td_error = td_target - q_table[state[0], state[1], action]
                q_table[state[0], state[1], action] += alpha * td_error

                action = choose_action(next_state, q_table, epsilon, env)

            elif algorithm == "sarsa":
                # On-policy update using actual next action selected
                next_action = choose_action(next_state, q_table, epsilon, env)
                td_target = reward + gamma * q_table[next_state[0], next_state[1], next_action] * (not terminated)
                td_error = td_target - q_table[state[0], state[1], action]
                q_table[state[0], state[1], action] += alpha * td_error

                action = next_action

            state = next_state

        step_history.append(total_steps)
        reward_history.append(episode_reward)

    return q_table, np.array(step_history), np.array(reward_history)

In [ ]:
# Train both agents
q_table_ql, steps_ql, rewards_ql = train_agent(algorithm="q_learning")
q_table_sarsa, steps_sarsa, rewards_sarsa = train_agent(algorithm="sarsa")

# Compute moving averages (smoothing)
window = 20
smooth_rewards_ql = np.convolve(rewards_ql, np.ones(window) / window, mode='valid')
smooth_steps_ql = steps_ql[window-1:]

smooth_rewards_sarsa = np.convolve(rewards_sarsa, np.ones(window) / window, mode='valid')
smooth_steps_sarsa = steps_sarsa[window-1:]

# Plot Mean Episode Return vs Environment Steps
plt.figure(figsize=(10, 6))
plt.plot(smooth_steps_ql, smooth_rewards_ql, label="Q-Learning", color="blue", linewidth=2)
plt.plot(smooth_steps_sarsa, smooth_rewards_sarsa, label="SARSA", color="orange", linewidth=2)

plt.title("Mean Episode Return vs. Total Environment Steps", fontsize=14, fontweight="bold")
plt.xlabel("Total Environment Steps", fontsize=12)
plt.ylabel(f"Mean Episode Return (Window = {window})", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=12)
plt.show()

$$\tiny{\text{This code was made with the help of Gemini AI.}}$$